# Relation Extraction using PubmedBERT

Trying out different ways to understand how BERT models predict relations and how much is the dependency on entities

Method 1. Finetuning BERT on BioRED dataset and evaluating in the standard way

Method 2. Masking entities completely by removing text 

In [ ]:
# ! pip install -U transformers
# ! pip install accelerate>=0.26.0
# ! pip install torch
# ! pip install datasets transformers

In [39]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer
from itertools import combinations
from transformers import AutoModelForSequenceClassification
from torch.utils.data import DataLoader

## BioRED Corpus

BioRED Paper: https://academic.oup.com/bib/article/23/5/bbac282/6645993 

+ Entity mentions: 20,419
+ Unique concept identifiers: 3869
+ Relations: 6503

+ Novel relaions among all annotated relations 69%

In [14]:
dataset = load_dataset("bigbio/biored")

for split in ['train', 'validation', 'test']:
    print(f"\n{split} examples: {len(dataset[split])}")

for split in ['train', 'validation', 'test']:
    print(f"\n{split} example:")
    for key, value in dataset[split][0].items():
        print(key, value)
    
# Build relation list
relation_types = set()
for split in ['train', 'validation', 'test']:
    for ex in dataset[split]:
        for rel in ex['relations']:
            relation_types.add(rel['type'])

relation_label_list = sorted(list(relation_types))
relation_label_list.append("no_relation")  # add negative class
print("\nBioRED relation types:", relation_label_list)

relation2id = {rel: idx for idx, rel in enumerate(relation_label_list)}

Using the latest cached version of the dataset since bigbio/biored couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'biored_source' at /home/ubuntu/.cache/huggingface/datasets/bigbio___biored/biored_source/1.0.0/18063bd2d68db98142aaf46df1a6436e066ae226b6d0fbe293bd452e7bcdce23 (last modified on Mon Oct  6 19:44:04 2025).



train examples: 400

validation examples: 100

test examples: 100

train example:
pmid 10491763
passages [{'type': 'title', 'text': ['Hepatocyte nuclear factor-6: associations between genetic variability and type II diabetes and between genetic variability and estimates of insulin secretion.'], 'offsets': [[0, 158]]}, {'type': 'abstract', 'text': ['The transcription factor hepatocyte nuclear factor (HNF)-6 is an upstream regulator of several genes involved in the pathogenesis of maturity-onset diabetes of the young. We therefore tested the hypothesis that variability in the HNF-6 gene is associated with subsets of Type II (non-insulin-dependent) diabetes mellitus and estimates of insulin secretion in glucose tolerant subjects.   We cloned the coding region as well as the intron-exon boundaries of the HNF-6 gene. We then examined them on genomic DNA in six MODY probands without mutations in the MODY1, MODY3 and MODY4 genes and in 54 patients with late-onset Type II diabetes by combined

In [ ]:
model_name = "microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext"
tokenizer = AutoTokenizer.from_pretrained(model_name)
max_seq_length = 256

## Preprocess Text

+ Create a relation ground truth labels
+ Generate all possible forms of entity pairs
+ Add entity markers
+ Determine the label between those entity markers

In [ ]:
def flatten_text(t):
    if isinstance(t, list):
        return " ".join(str(x) for x in t)
    return str(t)

def get_text_from_passages(passages):
    texts = []
    for p in passages:
        texts.append(flatten_text(p["text"]))
    return " ".join(texts)


# Preprocessing function
def preprocess_example(example):
    inputs = []

    # Flatten passages into a single string
    text = get_text_from_passages(example["passages"])

    # Map entity IDs to entity info
    entity_dict = {e["concept_id"]: e for e in example["entities"]}
    # print("Entity dict:", entity_dict)

    # Create gold relation pairs
    gold_pairs = {}
    for rel in example["relations"]:
        arg1 = rel["concept_1"]
        arg2 = rel["concept_2"]
        gold_pairs[(arg1, arg2)] = rel["type"]

    # Generate all candidate entity pairs
    entity_ids = [e["concept_id"] for e in example["entities"]]
    for e1_id, e2_id in combinations(entity_ids, 2):
        e1 = entity_dict[e1_id]
        e2 = entity_dict[e2_id]

        e1_text = flatten_text(e1["text"])
        e2_text = flatten_text(e2["text"])

        # Insert <e1> and <e2> markers
        marked_text = text
        if e1["offsets"][0][0] < e2["offsets"][0][0]:
            marked_text = marked_text.replace(e1_text, f"<e1>{e1_text}</e1>", 1)
            marked_text = marked_text.replace(e2_text, f"<e2>{e2_text}</e2>", 1)
        else:
            marked_text = marked_text.replace(e2_text, f"<e2>{e2_text}</e2>", 1)
            marked_text = marked_text.replace(e1_text, f"<e1>{e1_text}</e1>", 1)

        # Determine label
        label_str = gold_pairs.get((e1_id, e2_id),
                                   gold_pairs.get((e2_id, e1_id), "no_relation"))
        label_id = relation2id[label_str]
        # print(label_id)

        # Tokenize
        tokenized = tokenizer(
            marked_text,
            padding="max_length",
            truncation=True,
            max_length=max_seq_length,
            return_tensors="pt"
        )

        tokenized["labels"] = torch.tensor(label_id, dtype=torch.long)
        inputs.append(tokenized)

    return inputs


## Method 1

In [23]:
# Preprocess training set
train_inputs = []
for ex in dataset["train"]:
    train_inputs.extend(preprocess_example(ex))



## Total training pairs and tokenised training example

+ input ids are basically vocab id mapped to the word in the marked text since vocab is 256, it would be either truncated or padded by 0s
+ token type ids is all 0s because for RE our input is a single sentence unlike a QA task
+ attention mask 1 is non padded and 0 is padded, all are 1s since sentence length for first training sample is >256

In [24]:
print(f"Number of training examples: {len(train_inputs)}")
print(train_inputs[0].keys())  # should show 'input_ids', 'attention_mask', 'labels'

Number of training examples: 254924
KeysView({'input_ids': tensor([[    2, 14675,  4254,  2991,    17,    26,    30,  5253,  2192,  3299,
          5687,  1930,  2601,  2890,  4032,  1930,  2192,  3299,  5687,  1930,
          5280,  1927,  4122,  5512,    18,  1920,  3213,  2991, 14675,  4254,
          2991,    12, 18268,    13,    17,    26,  1977,  1925,  7671,  7888,
          1927,  2980,  2628,  3286,  1922,  1920,  6559,  1927, 17728,    17,
          4727,  4032,  1927,  1920,  4201,    18,  2038,  2955,  3620,  1920,
          4676,  1988,  5687,  1922,  1920,    32, 11123,    34, 18268,    17,
            26,    32,    19, 11123,    34,  2359,  1977,  2458,  1956,  9524,
          1927,  2601,  2890,    12,  2447,    17,  4122,    17,  3100,    13,
          4032,  9695,  1930,  5280,  1927,  4122,  5512,  1922,  3817, 16126,
          3297,    18,  2038,  8283,  1920,  5804,  3031,  1966,  2486,  1966,
          1920, 10444,    17,  7049, 11471,  1927,  1920, 18268,    17, 

In [26]:
from torch.utils.data import Dataset

class REDataset(Dataset):
    def __init__(self, inputs):
        self.inputs = inputs

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, idx):
        # Each element has 'input_ids', 'attention_mask', 'labels'
        item = self.inputs[idx]
        return {
            "input_ids": item["input_ids"].squeeze(),       # remove batch dim
            "attention_mask": item["attention_mask"].squeeze(),
            "labels": item["labels"]
        }

train_dataset = REDataset(train_inputs)
print(train_dataset[0])  # should show input_ids, attention_mask, labels tensors


{'input_ids': tensor([    2, 14675,  4254,  2991,    17,    26,    30,  5253,  2192,  3299,
         5687,  1930,  2601,  2890,  4032,  1930,  2192,  3299,  5687,  1930,
         5280,  1927,  4122,  5512,    18,  1920,  3213,  2991, 14675,  4254,
         2991,    12, 18268,    13,    17,    26,  1977,  1925,  7671,  7888,
         1927,  2980,  2628,  3286,  1922,  1920,  6559,  1927, 17728,    17,
         4727,  4032,  1927,  1920,  4201,    18,  2038,  2955,  3620,  1920,
         4676,  1988,  5687,  1922,  1920,    32, 11123,    34, 18268,    17,
           26,    32,    19, 11123,    34,  2359,  1977,  2458,  1956,  9524,
         1927,  2601,  2890,    12,  2447,    17,  4122,    17,  3100,    13,
         4032,  9695,  1930,  5280,  1927,  4122,  5512,  1922,  3817, 16126,
         3297,    18,  2038,  8283,  1920,  5804,  3031,  1966,  2486,  1966,
         1920, 10444,    17,  7049, 11471,  1927,  1920, 18268,    17,    26,
         2359,    18,  2038,  2759,  3572,  3581, 

In [31]:
num_labels = len(relation_label_list)
print("relation_label_list:", relation_label_list)
print("num_labels:", num_labels)

print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels
)
print(model.device)

relation_label_list: ['Association', 'Bind', 'Comparison', 'Conversion', 'Cotreatment', 'Drug_Interaction', 'Negative_Correlation', 'Positive_Correlation', 'no_relation']
num_labels: 9
GPU available: True
GPU name: NVIDIA RTX A6000


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


cpu


## Training BERT model

In [32]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./re_biobert",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    logging_steps=50
)

In [33]:
import transformers
print(transformers.__version__)


from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=train_dataset,   # replace with validation dataset if available
    tokenizer=tokenizer
)

4.57.0


/tmp/ipykernel_2245192/2815082000.py:7: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [38]:
#trainer.train()

In [34]:
finetuned_model_path = "re_biobert/checkpoint-47799"
model = AutoModelForSequenceClassification.from_pretrained(finetuned_model_path)
model.to("cuda" if torch.cuda.is_available() else "cpu")
model.eval()

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e

## Inference

In [36]:
def preprocess_example_for_inference(example):
    inputs = []
    text = get_text_from_passages(example["passages"])
    entity_dict = {e["concept_id"]: e for e in example["entities"]}
    entity_ids = [e["concept_id"] for e in example["entities"]]

    for e1_id, e2_id in combinations(entity_ids, 2):
        e1 = entity_dict[e1_id]
        e2 = entity_dict[e2_id]
        e1_text = flatten_text(e1["text"])
        e2_text = flatten_text(e2["text"])

        marked_text = text
        if e1["offsets"][0][0] < e2["offsets"][0][0]:
            marked_text = marked_text.replace(e1_text, f"<e1>{e1_text}</e1>", 1)
            marked_text = marked_text.replace(e2_text, f"<e2>{e2_text}</e2>", 1)
        else:
            marked_text = marked_text.replace(e2_text, f"<e2>{e2_text}</e2>", 1)
            marked_text = marked_text.replace(e1_text, f"<e1>{e1_text}</e1>", 1)

        tokenized = tokenizer(
            marked_text,
            padding="max_length",
            truncation=True,
            max_length=max_seq_length,
            return_tensors="pt"
        )
        tokenized["pair"] = (e1_id, e2_id)  # keep track of which pair this is
        inputs.append(tokenized)

    return inputs


## Preprocess Test Data

In [37]:
test_inputs = []
for ex in dataset["test"]:
    test_inputs.extend(preprocess_example_for_inference(ex))

print(f"Prepared {len(test_inputs)} test examples")

Prepared 72374 test examples


In [40]:
device = "cuda" if torch.cuda.is_available() else "cpu"

# Create DataLoader for efficient batching
batch_size = 16
def collate_fn(batch):
    return {
        "input_ids": torch.cat([x["input_ids"] for x in batch]).to(device),
        "attention_mask": torch.cat([x["attention_mask"] for x in batch]).to(device),
        "pairs": [x["pair"] for x in batch]
    }

test_loader = DataLoader(test_inputs, batch_size=batch_size, collate_fn=collate_fn)

predictions = []
with torch.no_grad():
    for batch in test_loader:
        outputs = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"]
        )
        probs = torch.softmax(outputs.logits, di=-1)
        preds = torch.argmax(probs, dim=-1).cpu().tolist()

        for pair, pred in zip(batch["pairs"], preds):
            predictions.append({
                "pair": pair,
                "pred_label_id": pred,
                "pred_label_name": relation_label_list[pred]
            })


## Predictions on Test Dataset

In [ ]:
import pandas as pd

df_preds = pd.DataFrame(predictions)
print(df_preds.head(20))

df_preds.to_csv("./data/biored_test_predictions.tsv", index=False, sep="\t")


                           pair  pred_label_id pred_label_name
0               (6331, D008133)              8     no_relation
1               (6331, D013610)              8     no_relation
2               (6331, D001919)              8     no_relation
3               (6331, D008133)              8     no_relation
4               (6331, D008133)              8     no_relation
5                  (6331, 9606)              8     no_relation
6               (6331, D001919)              8     no_relation
7               (6331, D054537)              8     no_relation
8               (6331, D017180)              8     no_relation
9               (6331, D054537)              8     no_relation
10              (6331, D008012)              8     no_relation
11              (6331, D008801)              8     no_relation
12              (6331, D017180)              8     no_relation
13              (6331, D008133)              8     no_relation
14                 (6331, 6331)              8     no_r

## Evaluation

In [46]:
from itertools import combinations
from sklearn.metrics import precision_recall_fscore_support, classification_report

gold_labels = []
pair_index = []  # same order as predictions

for ex in dataset["test"]:
    # Build gold mapping (concept1, concept2) -> relation type
    gold_pairs = {}
    for rel in ex["relations"]:
        gold_pairs[(rel["concept_1"], rel["concept_2"])] = rel["type"]

    # Build entity mapping
    entity_ids = [e["concept_id"] for e in ex["entities"]]
    for e1_id, e2_id in combinations(entity_ids, 2):
        label = gold_pairs.get((e1_id, e2_id),
                               gold_pairs.get((e2_id, e1_id), "no_relation"))
        gold_labels.append(relation2id[label])
        pair_index.append((e1_id, e2_id))


In [47]:
# Build dict for fast lookup
pred_dict = {tuple(p['pair']): p['pred_label_id'] for p in predictions}

# Align in same order as gold_labels
pred_labels = [pred_dict.get(pair, relation2id["no_relation"]) for pair in pair_index]


In [48]:
# Calculate precision, recall, F1 (macro + micro)
precision, recall, f1, _ = precision_recall_fscore_support(
    gold_labels, pred_labels, average='macro', zero_division=0
)
precision_micro, recall_micro, f1_micro, _ = precision_recall_fscore_support(
    gold_labels, pred_labels, average='micro', zero_division=0
)

print("=== Overall Evaluation ===")
print(f"Macro Precision: {precision:.4f}, Recall: {recall:.4f}, F1: {f1:.4f}")
print(f"Micro Precision: {precision_micro:.4f}, Recall: {recall_micro:.4f}, F1: {f1_micro:.4f}")


=== Overall Evaluation ===
Macro Precision: 0.5249, Recall: 0.5699, F1: 0.5136
Micro Precision: 0.7974, Recall: 0.7974, F1: 0.7974


In [50]:
print("\n Per Relation Type Stats")
print(classification_report(
    gold_labels, pred_labels,
    target_names=relation_label_list,
    digits=4,
    zero_division=0
))



 Per Relation Type Stats
                      precision    recall  f1-score   support

         Association     0.5195    0.4930    0.5059      9175
                Bind     0.1462    0.7333    0.2438        60
          Comparison     0.7313    0.7778    0.7538       252
          Conversion     0.0000    0.0000    0.0000         8
         Cotreatment     0.6500    0.2308    0.3406       169
    Drug_Interaction     0.8667    1.0000    0.9286       156
Negative_Correlation     0.4361    0.5081    0.4693      3503
Positive_Correlation     0.4701    0.4899    0.4798      4848
         no_relation     0.9045    0.8966    0.9005     54203

            accuracy                         0.7974     72374
           macro avg     0.5249    0.5699    0.5136     72374
        weighted avg     0.8019    0.7974    0.7990     72374



## Method 2

In [51]:
import torch
from itertools import combinations
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import precision_recall_fscore_support, classification_report


model_path = "re_biobert/checkpoint-47799"   # <-- your fine-tuned model path
max_seq_length = 256
device = "cuda" if torch.cuda.is_available() else "cpu"

# Load model + tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path).to(device)
model.eval()

# Load BioRED dataset
dataset = load_dataset("bigbio/biored")

# Build relation label set
relation_types = set()
for split in ['train', 'validation', 'test']:
    for ex in dataset[split]:
        for rel in ex['relations']:
            relation_types.add(rel['type'])

relation_label_list = sorted(list(relation_types))
relation_label_list.append("no_relation")
relation2id = {r: i for i, r in enumerate(relation_label_list)}
id2relation = {i: r for r, i in relation2id.items()}


def flatten_text(t):
    if isinstance(t, list):
        return " ".join(str(x) for x in t)
    return str(t)

def get_text_from_passages(passages):
    return " ".join(flatten_text(p["text"]) for p in passages)


def preprocess_example_masked(example):
    inputs = []
    text = get_text_from_passages(example["passages"])
    entity_dict = {e["concept_id"]: e for e in example["entities"]}
    gold_pairs = {(r["concept_1"], r["concept_2"]): r["type"] for r in example["relations"]}
    entity_ids = [e["concept_id"] for e in example["entities"]]

    for e1_id, e2_id in combinations(entity_ids, 2):
        e1, e2 = entity_dict[e1_id], entity_dict[e2_id]
        e1_text, e2_text = flatten_text(e1["text"]), flatten_text(e2["text"])

        # Mask entity mentions
        masked_text = text
        if e1["offsets"][0][0] < e2["offsets"][0][0]:
            masked_text = masked_text.replace(e1_text, "[E1]", 1)
            masked_text = masked_text.replace(e2_text, "[E2]", 1)
        else:
            masked_text = masked_text.replace(e2_text, "[E2]", 1)
            masked_text = masked_text.replace(e1_text, "[E1]", 1)

        label_str = gold_pairs.get((e1_id, e2_id),
                                   gold_pairs.get((e2_id, e1_id), "no_relation"))
        label_id = relation2id[label_str]

        tokenized = tokenizer(
            masked_text,
            padding="max_length",
            truncation=True,
            max_length=max_seq_length,
            return_tensors="pt"
        )
        tokenized["labels"] = torch.tensor(label_id)
        tokenized["pair"] = (e1_id, e2_id)
        inputs.append(tokenized)
    return inputs


test_inputs = []
for ex in dataset["test"]:
    test_inputs.extend(preprocess_example_masked(ex))

print(f"Prepared {len(test_inputs)} masked test examples")

from torch.utils.data import DataLoader

def collate_fn(batch):
    return {
        "input_ids": torch.cat([x["input_ids"] for x in batch]).to(device),
        "attention_mask": torch.cat([x["attention_mask"] for x in batch]).to(device),
        "labels": torch.tensor([x["labels"].item() for x in batch]),
        "pairs": [x["pair"] for x in batch],
    }

loader = DataLoader(test_inputs, batch_size=16, collate_fn=collate_fn)

pred_labels, gold_labels = [], []
with torch.no_grad():
    for batch in loader:
        outputs = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"]
        )
        preds = torch.argmax(outputs.logits, dim=-1).cpu().tolist()
        pred_labels.extend(preds)
        gold_labels.extend(batch["labels"].cpu().tolist())

p_macro, r_macro, f_macro, _ = precision_recall_fscore_support(
    gold_labels, pred_labels, average="macro", zero_division=0
)
p_micro, r_micro, f_micro, _ = precision_recall_fscore_support(
    gold_labels, pred_labels, average="micro", zero_division=0
)

print("\n=== Masked Entity Evaluation (Task 2a) ===")
print(f"Macro: Precision={p_macro:.4f}, Recall={r_macro:.4f}, F1={f_macro:.4f}")
print(f"Micro: Precision={p_micro:.4f}, Recall={r_micro:.4f}, F1={f_micro:.4f}")

print("\n=== Per-Relation Breakdown ===")
print(classification_report(
    gold_labels, pred_labels,
    target_names=relation_label_list,
    digits=4,
    zero_division=0
))


Using the latest cached version of the dataset since bigbio/biored couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'biored_source' at /home/ubuntu/.cache/huggingface/datasets/bigbio___biored/biored_source/1.0.0/18063bd2d68db98142aaf46df1a6436e066ae226b6d0fbe293bd452e7bcdce23 (last modified on Mon Oct  6 19:44:04 2025).


Prepared 72374 masked test examples

=== Masked Entity Evaluation (Task 2a) ===
Macro: Precision=0.2220, Recall=0.1683, F1=0.1824
Micro: Precision=0.7395, Recall=0.7395, F1=0.7395

=== Per-Relation Breakdown ===
                      precision    recall  f1-score   support

         Association     0.3937    0.2191    0.2815      9175
                Bind     0.0000    0.0000    0.0000        60
          Comparison     0.0000    0.0000    0.0000       252
          Conversion     0.0000    0.0000    0.0000         8
         Cotreatment     0.0000    0.0000    0.0000       169
    Drug_Interaction     0.0000    0.0000    0.0000       156
Negative_Correlation     0.4701    0.2375    0.3156      3503
Positive_Correlation     0.3476    0.1355    0.1950      4848
         no_relation     0.7868    0.9229    0.8494     54203

            accuracy                         0.7395     72374
           macro avg     0.2220    0.1683    0.1824     72374
        weighted avg     0.6852    0.7395 